## VIT Setup

In [ ]:
# imports
import os
import sqlite3
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from tqdm import tqdm
import sys
import itertools
import math
import gdown
from torch.cuda.amp import autocast, GradScaler

print(f"PyTorch version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
DB_PATH = './beatmaps.db'

try:
  import google.colab  # type: ignore
  if not os.path.exists('/content/beatmaps.db'):
    url = 'https://drive.google.com/uc?id=14yvshmHQ069SCjKIa8uBak8aPcyccSqv'
    output = '/content/beatmaps.db'
    gdown.download(url, output, quiet=False)
  DB_PATH = '/content/beatmaps.db'
except Exception:
    pass

print(f"Database path: {DB_PATH}")

MAX_SEQ_LEN = 1024
PATCH_SIZE = 8  
MAX_NUM_PATCHES = MAX_SEQ_LEN // PATCH_SIZE 
IN_CHANNELS = 6

D_MODEL = 512
N_HEADS = 8
N_LAYERS = 6
DIM_FEEDFORWARD = 4 * D_MODEL
DROPOUT = 0.1

BATCH_SIZE = 64

In [ ]:
def load_and_group_data_from_db_fast_optimized(db_path, chunk_size=1000):
    print("Connecting to database...")
    con = sqlite3.connect(db_path)
    cursor = con.cursor()

    print("Fetching valid beatmap IDs and all metadata...")
    metadata_df = pd.read_sql_query(
        """
        SELECT id, ar, od, circle_size as cs, main_bpm, difficulty_rating 
        FROM beatmaps 
        WHERE main_bpm IS NOT NULL AND difficulty_rating IS NOT NULL
        ORDER BY id
        """, 
        con
    )
    valid_map_ids = metadata_df['id'].tolist()
    
    metadata_dict = {
        row.id: np.array([row.ar, row.od, row.cs, row.main_bpm, row.difficulty_rating], dtype=np.float32)
        for row in metadata_df.itertuples(index=False)
    }
    
    print(f"Found {len(valid_map_ids)} beatmaps with complete metadata.")

    processed_data = []
    num_chunks = (len(valid_map_ids) + chunk_size - 1) // chunk_size

    vector_query_template = """
        SELECT beatmap_id, x_diff, y_diff, time_diff, length, abs_x, abs_y
        FROM beatmap_vectors 
        WHERE beatmap_id IN ({placeholders}) 
        ORDER BY beatmap_id
    """

    for i in tqdm(
        range(0, len(valid_map_ids), chunk_size),
        total=num_chunks,
        desc="Processing Chunks",
        dynamic_ncols=True,
        leave=True,
        file=sys.stdout
    ):
        chunk_ids = valid_map_ids[i:i + chunk_size]
        
        placeholders = ','.join('?' for _ in chunk_ids)
        query = vector_query_template.format(placeholders=placeholders)
        
        cursor.execute(query, chunk_ids)
        
        for map_pk, group_iter in itertools.groupby(cursor, key=lambda row: row[0]):
            
            vectors_list = [row[1:] for row in group_iter]
            
            if not vectors_list:
                continue

            meta_np = metadata_dict.get(map_pk)
            if meta_np is None:
                continue 

            vectors_tensor = torch.tensor(vectors_list, dtype=torch.float32)
            vectors_tensor[:, 2] = torch.log1p(vectors_tensor[:, 2])
            metadata_tensor = torch.from_numpy(meta_np) 
            
            processed_data.append((vectors_tensor, metadata_tensor))

    con.close()
    print("Finished processing all data.")
    return processed_data

all_beatmaps_data = load_and_group_data_from_db_fast_optimized(DB_PATH, chunk_size=1000)

METADATA_DIM = all_beatmaps_data[0][1].shape[0]
print(f"\nDetected metadata dimension: {METADATA_DIM}")

In [ ]:
from torch.utils.data import random_split

val_size = int(len(all_beatmaps_data) * 0.1)
train_size = len(all_beatmaps_data) - val_size
train_data, val_data = random_split(all_beatmaps_data, [train_size, val_size])

print(f"Data split into {len(train_data)} training samples and {len(val_data)} validation samples.")

print("\nCalculating normalization statistics from the augmented training set...")

all_vectors_list = [data[0] for data in train_data]
all_metadata_list = [data[1] for data in train_data]

augmented_vectors_list_for_stats = []
for vectors in all_vectors_list:
    augmented_vectors_list_for_stats.append(vectors)
    
    flipped_x = vectors.clone()
    flipped_x[:, 0] *= -1                  
    flipped_x[:, 4] = 512 - flipped_x[:, 4]  
    augmented_vectors_list_for_stats.append(flipped_x)
    
    flipped_y = vectors.clone()
    flipped_y[:, 1] *= -1                  
    flipped_y[:, 5] = 384 - flipped_y[:, 5]  
    augmented_vectors_list_for_stats.append(flipped_y)

    flipped_xy = vectors.clone()
    flipped_xy[:, 0] *= -1                  
    flipped_xy[:, 1] *= -1                  
    flipped_xy[:, 4] = 512 - flipped_xy[:, 4]  
    flipped_xy[:, 5] = 384 - flipped_xy[:, 5]  
    augmented_vectors_list_for_stats.append(flipped_xy)

all_vectors_tensor = torch.cat(augmented_vectors_list_for_stats, dim=0)
all_metadata_tensor = torch.stack(all_metadata_list, dim=0) 

vector_mean = all_vectors_tensor.mean(dim=0)
vector_std = all_vectors_tensor.std(dim=0)

meta_mean = all_metadata_tensor.mean(dim=0)
meta_std = all_metadata_tensor.std(dim=0)

vector_std[vector_std == 0] = 1.0
meta_std[meta_std == 0] = 1.0

print("\n--- Normalization Stats (from augmented training data) ---")
print(f"Vector Mean: {vector_mean.numpy()}")
print(f"Vector Std:  {vector_std.numpy()}")
print(f"Meta Mean:   {meta_mean.numpy()}")
print(f"Meta Std:    {meta_std.numpy()}")

In [ ]:
def collate_fn_patched(batch, max_num_patches, patch_size, vector_dim):
    """
    Collates a batch of variable-length sequences into fixed-size patched tensors.
    Patches are NOT flattened to preserve intra-patch sequential information.
    """
    vectors, metadata = zip(*batch)
    
    padded_patches = torch.zeros(len(batch), max_num_patches, patch_size, vector_dim, dtype=torch.float32)
    attention_mask = torch.zeros(len(batch), max_num_patches, dtype=torch.bool)
    
    for i, v in enumerate(vectors):
        max_len = max_num_patches * patch_size
        if v.shape[0] > max_len:
            v = v[:max_len]
            
        num_vectors = v.shape[0]
        num_patches = math.ceil(num_vectors / patch_size)
        
        padding_needed = num_patches * patch_size - num_vectors
        if padding_needed > 0:
            padding = torch.zeros(padding_needed, vector_dim, dtype=v.dtype, device=v.device)
            v = torch.cat([v, padding], dim=0)

        patches = v.view(num_patches, patch_size, vector_dim)
        
        padded_patches[i, :num_patches] = patches
        attention_mask[i, :num_patches] = True
        
    stacked_metadata = torch.stack(metadata, dim=0)
    
    return padded_patches.to(device), attention_mask.to(device), stacked_metadata.to(device)

class BeatmapDataset(Dataset):
    """
    PyTorch Dataset for osu! beatmaps.
    Handles data loading, on-the-fly augmentation, and normalization.
    Returns variable-length sequences.
    """
    def __init__(self, beatmap_data, vector_mean, vector_std, meta_mean, meta_std, augment=False):
        self.beatmap_data = beatmap_data
        self.vector_mean = vector_mean
        self.vector_std = vector_std
        self.meta_mean = meta_mean
        self.meta_std = meta_std
        self.epsilon = 1e-8
        self.augment = augment
        self.num_augmentations = 4 if self.augment else 1

    def __len__(self):
        return len(self.beatmap_data) * self.num_augmentations

    def __getitem__(self, idx):
        original_idx = idx // self.num_augmentations
        aug_type = idx % self.num_augmentations

        vectors, metadata = self.beatmap_data[original_idx]

        vectors = vectors.clone()
        if self.augment:
            if aug_type == 1: 
                vectors[:, 0] *= -1                  
                vectors[:, 4] = 512 - vectors[:, 4]  
            elif aug_type == 2:
                vectors[:, 1] *= -1                  
                vectors[:, 5] = 384 - vectors[:, 5]  
            elif aug_type == 3:
                vectors[:, 0:2] *= -1                
                vectors[:, 4] = 512 - vectors[:, 4]  
                vectors[:, 5] = 384 - vectors[:, 5]  

        normalized_metadata = (metadata - self.meta_mean) / (self.meta_std + self.epsilon)
        normalized_vectors = (vectors - self.vector_mean) / (self.vector_std + self.epsilon)

        return normalized_vectors, normalized_metadata

train_dataset = BeatmapDataset(train_data, vector_mean, vector_std, meta_mean, meta_std, augment=True)
val_dataset = BeatmapDataset(val_data, vector_mean, vector_std, meta_mean, meta_std, augment=False)

collate_with_args_patched = lambda batch: collate_fn_patched(
    batch, 
    max_num_patches=MAX_NUM_PATCHES, 
    patch_size=PATCH_SIZE, 
    vector_dim=IN_CHANNELS
)

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_with_args_patched)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_with_args_patched)

print(f"Created a training dataset with {len(train_dataset)} samples (including augmentations).")
print(f"Created a validation dataset with {len(val_dataset)} samples.")
print("\nA sample batch will be a tuple: (patched_vectors, attention_mask, metadata)")
sample_patches, sample_mask, sample_meta = next(iter(train_dataloader))
print(f"Patches shape: {sample_patches.shape}, Mask shape: {sample_mask.shape}, Meta shape: {sample_meta.shape}")
print(f"Expected patches shape: (batch_size, num_patches, patch_size, in_channels)")

In [ ]:
class PatchedOsuBert(nn.Module):
    def __init__(self, *, max_num_patches, patch_size, d_model, n_heads, n_layers, dim_feedforward, dropout, metadata_dim, in_channels):
        super().__init__()
        self.d_model = d_model
        self.in_channels = in_channels
        
        self.patch_embedder = nn.Sequential(
            nn.Conv1d(in_channels, d_model, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1)
        )
        
        self.metadata_proj = nn.Linear(metadata_dim, d_model)
        
        self.metadata_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.randn(1, max_num_patches + 1, d_model))
        self.embed_dropout = nn.Dropout(dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=n_heads, 
            dim_feedforward=dim_feedforward, 
            dropout=dropout, 
            activation='gelu', 
            batch_first=True, 
            norm_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x_patched, metadata, attention_mask):
        batch_size, num_patches, patch_len, _ = x_patched.shape
        
        x_for_conv = x_patched.view(-1, patch_len, self.in_channels).permute(0, 2, 1)
        x_embed_flat = self.patch_embedder(x_for_conv).squeeze(-1)
        x_embed = x_embed_flat.view(batch_size, num_patches, -1)
        
        meta_embed = self.metadata_proj(metadata).unsqueeze(1) + self.metadata_token
        
        full_embeddings = torch.cat([meta_embed, x_embed], dim=1)
        full_embeddings += self.pos_embed
        full_embeddings = self.embed_dropout(full_embeddings)
        
        meta_mask = torch.zeros((batch_size, 1), dtype=torch.bool, device=x_patched.device)
        padding_mask = ~attention_mask
        full_padding_mask = torch.cat([meta_mask, padding_mask], dim=1)
        
        output = self.transformer_encoder(full_embeddings, src_key_padding_mask=full_padding_mask)
        output = self.norm(output)
        
        return output

model = PatchedOsuBert(
    max_num_patches=MAX_NUM_PATCHES,
    patch_size=PATCH_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    in_channels=IN_CHANNELS,
    metadata_dim=METADATA_DIM
).to(device)

with torch.no_grad():
    output = model(sample_patches, sample_meta, sample_mask)

print("--- Patched Model Sanity Check ---")
print(f"Input patches shape:  {sample_patches.shape}")
print(f"Input mask shape:     {sample_mask.shape}")
print(f"Output tensor shape:  {output.shape}")

expected_shape = (BATCH_SIZE, MAX_NUM_PATCHES + 1, D_MODEL)
print(f"Expected output shape: {expected_shape}")
assert output.shape == expected_shape, "Mismatch between output and expected shape!"

print("\n✅ Sanity check passed! The patch-based architecture is working.")
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {num_params / 1e6:.2f}M")

## MLM Training

In [ ]:
import torch.nn.functional as F

class LightweightDecoder(nn.Module):
    """
    A lightweight Transformer decoder.
    ASSUMES the input tensor 'x' is already projected to decoder_dim.
    """
    def __init__(self, decoder_dim, n_heads, n_layers, patch_dim, dropout, max_num_patches):
        super().__init__()
        
        self.pos_embed = nn.Parameter(torch.randn(1, max_num_patches, decoder_dim))
        
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=decoder_dim,
            nhead=n_heads,
            dim_feedforward=4 * decoder_dim,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.decoder_blocks = nn.TransformerEncoder(decoder_layer, num_layers=n_layers)
        self.decoder_norm = nn.LayerNorm(decoder_dim)
        self.prediction_head = nn.Linear(decoder_dim, patch_dim)

    def forward(self, x):
        x = x + self.pos_embed
        x = self.decoder_blocks(x)
        x = self.decoder_norm(x)
        predictions = self.prediction_head(x)
        return predictions

class PatchedOsuBertForMLM(nn.Module):
    def __init__(self, bert_model, patch_size, in_channels, masking_ratio=0.75,
                 decoder_dim=128, decoder_layers=4, decoder_heads=4):
        super().__init__()
        self.bert = bert_model
        self.masking_ratio = masking_ratio
        self.decoder_dim = decoder_dim
        
        patch_dim = patch_size * in_channels

        self.decoder_embed = nn.Linear(bert_model.d_model, decoder_dim)
        
        self.mask_token = nn.Parameter(torch.randn(1, 1, decoder_dim))

        self.decoder = LightweightDecoder(
            decoder_dim=decoder_dim,
            n_heads=decoder_heads,
            n_layers=decoder_layers,
            patch_dim=patch_dim,
            dropout=0.1,
            max_num_patches=bert_model.pos_embed.shape[1] - 1
        )

    def forward(self, x_patched, metadata, attention_mask):
        batch_size, num_patches, patch_len, _ = x_patched.shape
        
        x_for_conv = x_patched.view(-1, patch_len, self.bert.in_channels).permute(0, 2, 1)
        x_embed_flat = self.bert.patch_embedder(x_for_conv).squeeze(-1)
        x_embed = x_embed_flat.view(batch_size, num_patches, -1)
        
        num_visible = int(num_patches * (1 - self.masking_ratio))
        noise = torch.rand(batch_size, num_patches, device=x_patched.device)
        
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_visible = ids_shuffle[:, :num_visible]
        ids_masked = ids_shuffle[:, num_visible:]

        mask_bool = torch.zeros(batch_size, num_patches, dtype=torch.bool, device=x_patched.device)
        mask_bool.scatter_(1, ids_masked, True)
        
        visible_patches_embed = torch.gather(x_embed, 1, ids_visible.unsqueeze(-1).expand(-1, -1, self.bert.d_model))
        
        meta_embed = self.bert.metadata_proj(metadata).unsqueeze(1) + self.bert.metadata_token
        encoder_input = torch.cat([meta_embed, visible_patches_embed], dim=1)

        pos_embed_expanded = self.bert.pos_embed.expand(batch_size, -1, -1)
        visible_pos_embed = torch.gather(pos_embed_expanded, 1, torch.cat([
            torch.zeros(batch_size, 1, dtype=torch.long, device=x_patched.device),
            ids_visible + 1
        ], dim=1).unsqueeze(-1).expand(-1, -1, self.bert.d_model))
        
        encoder_input = encoder_input + visible_pos_embed
        
        encoded_tokens = self.bert.transformer_encoder(encoder_input)
        encoded_tokens = self.bert.norm(encoded_tokens)

        encoded_meta, encoded_visible_patches = encoded_tokens[:, :1], encoded_tokens[:, 1:]

        decoder_visible_embed = self.decoder_embed(encoded_visible_patches)
        
        mask_tokens = self.mask_token.repeat(batch_size, ids_masked.shape[1], 1)
        
        full_sequence_for_decoder = torch.cat([decoder_visible_embed, mask_tokens], dim=1)
        full_sequence_for_decoder = torch.gather(full_sequence_for_decoder, 1, ids_restore.unsqueeze(-1).expand(-1, -1, self.decoder_dim))
        
        decoded_patches = self.decoder(full_sequence_for_decoder)
        # ---------------------------------------------------------

        target_patches = x_patched.view(batch_size, num_patches, -1)
        
        loss = F.mse_loss(decoded_patches[mask_bool], target_patches[mask_bool])
        
        return loss

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import math
import os
from tqdm.auto import tqdm

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.05
NUM_EPOCHS = 5
total_steps = len(train_dataloader) * NUM_EPOCHS
WARMUP_RATIO = 0.05
warmup_steps = int(WARMUP_RATIO * total_steps)
MIN_LR = 1e-6
base_lr = LEARNING_RATE
eta_min = MIN_LR
device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = (device == 'cuda')

CHECKPOINT_DIR = "./checkpoints"
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "mlm_patched_bert_latest.pth")
os.makedirs(CHECKPOINT_DIR, exist_ok=True) 

bert_encoder_patched = PatchedOsuBert(
    max_num_patches=MAX_NUM_PATCHES,
    patch_size=PATCH_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    in_channels=IN_CHANNELS,
    metadata_dim=METADATA_DIM
)

mlm_model = PatchedOsuBertForMLM(
    bert_encoder_patched, 
    patch_size=PATCH_SIZE, 
    in_channels=IN_CHANNELS, 
    masking_ratio=0.25
).to(device)


optimizer = torch.optim.AdamW(mlm_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

def lr_lambda(current_step: int):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = (current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
    return (eta_min / base_lr) + (1.0 - (eta_min / base_lr)) * cosine_decay

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

start_epoch = 0
if os.path.exists(CHECKPOINT_PATH):
    print(f"--- Found checkpoint at {CHECKPOINT_PATH}. Loading... ---")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    
    mlm_model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    
    print(f"--- Resuming training from Epoch {start_epoch + 1} ---")
else:
    print("--- No checkpoint found. Starting training from scratch. ---")

print("\n--- Starting Patched BERT Pre-training (ViT-style MLM) ---")
print(f"Device: {device}, AMP Enabled: {use_amp}")
print(f"Total Epochs: {NUM_EPOCHS}, Starting from Epoch: {start_epoch + 1}")
print(f"Batch Size: {BATCH_SIZE}, Base LR: {LEARNING_RATE}")
print(f"Patch Size: {PATCH_SIZE}, Max Patches: {MAX_NUM_PATCHES}")
print(f"Total training steps: {total_steps}, Warmup steps: {warmup_steps}")
print("-" * 50)

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start_time = time.time()
    
    mlm_model.train()
    train_loss = 0.0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]", dynamic_ncols=True)
    
    for patched_vectors, attention_mask, metadata in progress_bar:
        optimizer.zero_grad()
        
        with torch.amp.autocast(device_type=device, enabled=use_amp):
            loss = mlm_model(patched_vectors, metadata, attention_mask)
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        train_loss += loss.item()
        progress_bar.set_postfix({
            "Loss": f"{loss.item():.4f}",
            "LR": f"{optimizer.param_groups[0]['lr']:.2e}"
        })
        
    avg_train_loss = train_loss / len(train_dataloader)

    mlm_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for patched_vectors, attention_mask, metadata in val_dataloader:
            with torch.amp.autocast(device_type=device, enabled=use_amp):
                loss = mlm_model(patched_vectors, metadata, attention_mask)
                val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_dataloader) if len(val_dataloader) > 0 else 0.0
    
    epoch_duration = time.time() - epoch_start_time
    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.2e} | "
        f"Time: {epoch_duration:.2f}s"
    )

    # --- Checkpoint Saving ---
    checkpoint_data = {
        'epoch': epoch,
        'model_state_dict': mlm_model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'val_loss': avg_val_loss,
    }
    torch.save(checkpoint_data, CHECKPOINT_PATH)
    print(f"Checkpoint saved for epoch {epoch+1} to {CHECKPOINT_PATH}")
    print("-" * 50)


print("\n--- Training Finished ---")